<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch12_multiple_comparisons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第12回：多重比較・事後推定・効果量（emmeans）


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Gelman & Stern (2006) 「有意と非有意の差は、それ自体は有意ではない」／Lenth, *emmeans* パッケージ。

In [ ]:
install.packages(c("lme4", "lmerTest", "emmeans"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(emmeans)   # 推定周辺平均・多重比較・効果量の標準的なパッケージ
library(tidyverse)


■ モデルを当てはめたあとに残る仕事：

第5回までで学んだのは「モデルを当てはめる」ところまでだった。しかし研究の問いは、たいてい係数そのものではなく**「どの水準とどの水準が違うのか」**という形をしている。調音位置が3か所あれば、知りたいのは3通りの対比較であって、`anova()` が返す「どこかに差がある」という1つのF値ではない。

この段階で2つの落とし穴が待っている。1つは**多重比較**——比較の回数を増やせば、偶然の有意が紛れ込む確率が上がる。もう1つは**「有意差の有無の差」を効果の差と取り違える**ことである。本章はこの2つを実データで確認し、`emmeans` による正しい扱い方を学ぶ。

■ 3水準以上の要因を扱う：

In [ ]:
set.seed(31)
n_speaker <- 16
n_word    <- 30
places <- c("labial", "alveolar", "velar")

vot_place <- expand_grid(speaker = sprintf("S%02d", 1:n_speaker),
                         word = sprintf("W%02d", 1:n_word)) %>%
  mutate(
    # 語を3つの調音位置に均等に割り当てる
    place = factor(places[((as.integer(sub("W", "", word)) - 1) %% 3) + 1], levels = places),
    si = as.integer(factor(speaker)),
    wi = as.integer(factor(word)),
    # 真の平均: labial 55, alveolar 58.2, velar 75
    # labialとalveolarの差は3.2msと小さく、velarだけが大きく離れている
    VOT = c(labial = 55, alveolar = 58.2, velar = 75)[as.character(place)] +
          rnorm(n_speaker, 0, 8)[si] + rnorm(n_word, 0, 4)[wi] + rnorm(n(), 0, 7)
  ) %>%
  select(-si, -wi)

model_place <- lmer(VOT ~ place + (1 | speaker) + (1 | word), data = vot_place)
anova(model_place)

# 推定周辺平均（estimated marginal means）: 各水準の「モデルに基づく平均」
em <- emmeans(model_place, ~ place)
em


■ ★よくある誤り その1：対比較を調整せずに繰り返す：

3水準あれば対比較は3通り、4水準なら6通り、5水準なら10通りに増える。1回あたり5%の誤りを許す検定を10回繰り返せば、どれか1つが偶然有意になる確率は40%に達する。調整の有無で結論がどう変わるかを実際に見る。

In [ ]:
cat("---- 調整なし（各比較を独立に検定）----\n")
print(pairs(em, adjust = "none"))

cat("\n---- Tukey法で調整（既定）----\n")
print(pairs(em))

cat("\n---- Bonferroni法で調整（保守的）----\n")
print(pairs(em, adjust = "bonferroni"))

# --- 実行結果（p値の部分だけ抜粋）---
#  contrast          estimate   SE df t.ratio  調整なし    Tukey   Bonferroni
#  labial - alveolar    -3.61 1.51 27  -2.390   0.0241    0.0605     0.0723
#  labial - velar      -17.70 1.51 27 -11.719  <0.0001   <0.0001    <0.0001
#  alveolar - velar    -14.09 1.51 27  -9.329  <0.0001   <0.0001    <0.0001
#
# 【出力の読み方】
# ・**labial対alveolarの結論が、調整の有無でひっくり返る。**
#   調整しなければ p = 0.0241 で「有意差あり」、Tukey法なら p = 0.0605、
#   Bonferroni法なら p = 0.0723 で「有意差なし」になる。
#   推定値（-3.61ms）もt値（-2.39）も同じで、**変わったのはp値だけ**である。
# ・他の2つの比較はどの方法でも p < 0.0001 で、調整しても揺るがない。
#   **調整が結論を変えるのは、もともと境界線上にある比較だけ**である。
# ・Tukey法とBonferroni法の違いは、Tukeyが「すべての対比較を行う」という
#   状況に特化して作られているのに対し、Bonferroniは比較の数で単純に割る
#   汎用的な方法である点にある。したがって**対比較が目的ならTukeyのほうが
#   検出力が高い**（0.0605 < 0.0723）。Bonferroniは、比較の内容が
#   対比較に限らない場合や、事前に決めた少数の比較だけを行う場合に使う。
# ・★では常に調整すべきかというと、そうではない。**事前に1つの比較だけを
#   仮説として決めていたなら、調整は不要である。** 多重比較の問題は
#   「たくさん試して当たったものを報告する」ことから生じるのであって、
#   比較の回数そのものが罪なのではない。研究計画の段階で
#   「どの比較を見るか」を決めておけば、この問題の多くは回避できる。
# ・★逆に最も避けるべきは、**全部の比較を見てから、有意だったものだけを
#   「仮説だった」として報告する**ことである。これは調整の有無以前の問題で、
#   検定の意味そのものを失わせる。


■ 効果量を報告する：

p値は「効果があるか」にしか答えず、「どれくらいか」には答えない。サンプルサイズを増やせばどんな小さな差でも有意になるため、**p値だけの報告では、実質的に意味のある差かどうかが分からない**。

In [ ]:
# Cohen's d 相当の標準化された効果量。残差SDを基準に差を割ったもの。
eff_size(em, sigma = sigma(model_place), edf = df.residual(model_place))

# --- 実行結果 ---
#  contrast          effect.size    SE   df lower.CL upper.CL
#  labial - alveolar      -0.508 0.213 23.1   -0.949   -0.067
#  labial - velar         -2.491 0.227 23.1   -2.960  -2.0210
#  alveolar - velar       -1.983 0.222 23.1   -2.443  -1.5240
#
# 【出力の読み方】
# ・effect.size は「差が残差SDの何個分か」を表す標準化された値である。
#   慣習的な目安は 0.2 で小、0.5 で中、0.8 で大とされる（Cohenの基準）。
# ・labial対alveolarは -0.508 で中程度。**p値では有意にならなかった比較だが、
#   効果量としては無視できる大きさではない。** これは
#   「有意でない＝効果が無い」ではないことの具体例である。
#   信頼区間が -0.949 から -0.067 と、0をわずかに外れる位置にあり、
#   **効果の大きさについてはまだ精度が足りない**と読むのが正しい。
# ・labial対velarは -2.491 で非常に大きい。VOTでいえば17.7msの差であり、
#   調音位置がVOTに与える影響としては音声学的にも妥当な大きさである。
# ・★効果量を報告する際は、**標準化された値と元の単位の値を両方**書くのがよい。
#   「-17.7ms（d = -2.49）」のように書けば、音声学の読者は17.7msという
#   物理量から意味を判断でき、他分野の読者は標準化された値で比較できる。
# ・★標準化の分母に何を使うかで値が変わる点に注意する。混合効果モデルでは
#   残差SDだけを使うか、ランダム効果の分散も含めるかで結果が変わる。
#   上では sigma(model_place)（残差SD）を使っている。
#   **どの分母を使ったかを明記しないと、他の研究と比較できない。**


■ ★よくある誤り その2：「一方は有意、他方は有意でない」を差だと考える：

音声研究で最も頻繁に見かける推論の誤りが、これである。「母語話者では効果が有意だったが、学習者では有意でなかった。したがって母語話者と学習者では効果が異なる」——この推論は成り立たない。Gelman & Stern (2006) が指摘したとおり、**2つの検定結果の違いは、それ自体が検定されていない**。実際に確かめる。

In [ ]:
set.seed(45)
n_sp <- 20

split_data <- expand_grid(speaker = sprintf("S%02d", 1:n_sp), item = 1:24) %>%
  mutate(
    group = if_else(as.integer(sub("S", "", speaker)) <= 10, "native", "learner"),
    cond  = rep(c("a", "b"), length.out = n()),
    # ★条件の効果は両群とも 4.0 に固定してある。つまり**群による違いは無い**。
    y = 50 + if_else(cond == "b", 4.0, 0) +
        rnorm(n_sp, 0, 6)[as.integer(factor(speaker))] + rnorm(n(), 0, 11)
  )

# ---- 誤った手続き: 群ごとに別々のモデルを当てはめて、有意かどうかを見る ----
for (g in c("native", "learner")) {
  mm <- lmer(y ~ cond + (1 | speaker), data = filter(split_data, group == g))
  cf <- coef(summary(mm))["condb", ]
  cat(sprintf("%-8s: beta = %.2f   t = %.2f   p = %.4f\n", g, cf[1], cf[4], cf[5]))
}

# ---- 正しい手続き: 交互作用として1つのモデルで直接検定する ----
model_int <- lmer(y ~ cond * group + (1 | speaker), data = split_data)
cf <- coef(summary(model_int))["condb:groupnative", ]
cat(sprintf("\n交互作用の直接検定: beta = %.3f   t = %.3f   p = %.4f\n",
            cf[1], cf[4], cf[5]))

# emmeansなら、群ごとの効果（単純効果）とその差を一度に出せる
cat("\n群ごとの条件の効果（単純効果）:\n")
print(pairs(emmeans(model_int, ~ cond | group)))

# --- 実行結果 ---
# native  : beta = 1.45   t = 1.02   p = 0.3093
# learner : beta = 4.14   t = 2.80   p = 0.0055
#
# 交互作用の直接検定: beta = -2.684   t = -1.308   p = 0.1915
#
# 【出力の読み方】
# ・別々に分析すると、学習者では p = 0.0055 で有意、母語話者では p = 0.3093 で
#   有意でない。ここから「学習者にだけ効果がある」と結論したくなる。
# ・**しかし交互作用を直接検定すると p = 0.1915 で、
#   2群の効果が違うという証拠は無い。** そして生成コードを見れば分かるとおり、
#   **真の効果は両群とも 4.0 で完全に同じ**である。
#   「学習者にだけ効果がある」という結論は誤りだった。
# ・なぜこうなるのか。母語話者の推定値は1.45、学習者は4.14で、確かに違って
#   見える。しかし各群10名ずつしかいないので、**それぞれの推定値が
#   大きな誤差を含んでいる**。片方がたまたま0.05の線の内側に、
#   もう片方が外側に落ちただけである。
# ・決定的な点は、**0.05という線のすぐ両側にある2つの値は、互いにほとんど
#   違わない**ということである。p = 0.049 と p = 0.051 は実質的に同じ証拠の
#   強さを持つのに、「有意/有意でない」という二分法はそれを正反対に見せる。
# ・正しい手続きは単純である。**群による違いを主張したいなら、
#   交互作用項を入れた1つのモデルで直接検定する。**
#   `emmeans(model, ~ cond | group)` を使えば、群ごとの単純効果も
#   同時に示せるので、記述と検定を両立できる。
# ・★この誤りは、学習者群と母語話者群、方言Aと方言B、若年層と高齢層など、
#   **群を比べる音声研究のほぼすべてに関わる**。査読でも指摘されやすい点なので、
#   「群ごとに分析して片方だけ有意だった」と書きそうになったら、
#   必ず交互作用の検定に置き換えること。


■ 演習：

1. `emmeans` で `place` の水準を2つずつではなく、「velar 対 それ以外の平均」という対比を作って検定せよ（ヒント: `contrast()` に係数ベクトルを渡す）。
2. 調整方法を `holm` に変えて結果を比べ、Bonferroniとの違いを説明せよ。
3. `split_data` で交互作用モデルの信頼区間を求め、「差が無い」と「差が分からない」の違いを論じよ。

■ 演習の解答：

In [ ]:
# 1. 独自の対比を作る。係数は emmeans の水準の順序に対応させる。
#    水準は labial, alveolar, velar の順なので、
#    「velar 対 他2つの平均」は c(-1/2, -1/2, 1) と書ける（合計0になること）。
contrast(em, list(velar_vs_rest = c(-1/2, -1/2, 1)))

# 2. Holm法はBonferroniを改良したもので、常にBonferroni以上の検出力を持つ。
cat("\n")
print(pairs(em, adjust = "holm"))

# 3. 交互作用の信頼区間を見る。
ci <- confint(model_int)["condb:groupnative", ]
cat("\n交互作用の95%信頼区間:", round(ci[1], 3), "から", round(ci[2], 3), "\n")
cat("生成時の真の交互作用: 0\n")

# --- 実行結果（抜粋）---
#  contrast      estimate   SE df t.ratio p.value
#  velar_vs_rest     15.9 1.31 27  12.152 <0.0001
#
#  contrast          estimate   SE df t.ratio p.value   ← holm法
#  labial - alveolar    -3.61 1.51 27  -2.390  0.0241
#  labial - velar      -17.70 1.51 27 -11.719 <0.0001
#  alveolar - velar    -14.09 1.51 27  -9.329 <0.0001
#
# 交互作用の95%信頼区間: -6.706 から 1.337
# 生成時の真の交互作用: 0
#
# 【出力の読み方】
# ・1問目: 「velar 対 他2つの平均」という対比の推定値は 15.9ms、
#   t = 12.152 で明確に有意である。係数 c(-1/2, -1/2, 1) は
#   **合計が0**になっており、これが対比（contrast）の条件である。
#   合計が0でないと、水準間の比較ではなく全体の水準の話になってしまう。
#   ★このように**理論的に意味のある対比を1つだけ事前に決めておけば、
#   3通りの対比較を全部行う必要が無く、多重比較の問題も小さくなる。**
#   「調音位置が後ろになるほどVOTが長い」という仮説があるなら、
#   対比較を並べるより、この形の対比のほうが仮説に直接答えている。
# ・2問目: Holm法では labial - alveolar が p = 0.0241 となり、
#   **調整なしのときと同じ値で、有意のまま**である。
#   一方 Bonferroni法では 0.0723、Tukey法では 0.0605 で有意にならなかった。
#   同じデータ・同じ推定値なのに、調整方法によって結論が3通りに分かれている。
# ・Holm法の仕組みを知ると理由が分かる。p値を小さい順に並べ、
#   最小のものには比較数（3）を、次には2を、最大のものには1を掛ける。
#   labial - alveolar は3つのうち最大のp値なので**1倍、つまり調整されない**。
#   Bonferroniが全部に3を掛けるのに比べ、Holmは常に緩い（＝検出力が高い）。
#   しかも第一種の過誤を全体で5%に抑える保証は同じである。
#   **したがってBonferroniを使う理由は、実はほとんど無い。**
# ・★ここから引き出すべき教訓は、「どの調整法が正しいか」ではなく、
#   **調整法はデータを見る前に決めておく**ということである。
#   3通り試して有意になったものを採るなら、調整した意味が消える。
# ・3問目: 交互作用の95%信頼区間は −6.706 から 1.337 で、0を含む。
#   だから「有意でない」となる。しかし区間の幅は8ポイント以上あり、
#   **−6.7という大きな差も、+1.3という逆向きの差も、まだ否定できていない**。
#   これは「差が無い」ことの証拠ではなく、**「差の大きさが分からない」**という
#   状態である。この2つは決定的に違う。
# ・「差が無い」と主張したいなら、信頼区間が**意味のある差の範囲を除外**して
#   いる必要がある。たとえば「3ms以上の差があれば意味がある」と決めているなら、
#   区間が −3 から +3 の内側に収まって初めて「差は無視できる」と言える。
#   このような手続きを同等性検定（equivalence testing）と呼ぶ。
#   ★p > 0.05 を「差が無い」と書くのは、本章で扱った誤りと同じ根を持つ。
